In [18]:
import pandas as pd
import numpy as np
import kagglehub
import os

path = kagglehub.dataset_download("uciml/glass")
df = pd.read_csv(os.path.join(path, "glass.csv"))

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())

Using Colab cache for faster access to the 'glass' dataset.
Dataset shape: (214, 10)
Columns: ['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type']
        RI     Na    Mg    Al     Si     K    Ca   Ba   Fe  Type
0  1.52101  13.64  4.49  1.10  71.78  0.06  8.75  0.0  0.0     1
1  1.51761  13.89  3.60  1.36  72.73  0.48  7.83  0.0  0.0     1
2  1.51618  13.53  3.55  1.54  72.99  0.39  7.78  0.0  0.0     1
3  1.51766  13.21  3.69  1.29  72.61  0.57  8.22  0.0  0.0     1
4  1.51742  13.27  3.62  1.24  73.08  0.55  8.07  0.0  0.0     1


In [19]:
df["y"] = (df["Type"] == 1).astype(int)
df = df.drop(columns=["Type"])

print(df.head())

        RI     Na    Mg    Al     Si     K    Ca   Ba   Fe  y
0  1.52101  13.64  4.49  1.10  71.78  0.06  8.75  0.0  0.0  1
1  1.51761  13.89  3.60  1.36  72.73  0.48  7.83  0.0  0.0  1
2  1.51618  13.53  3.55  1.54  72.99  0.39  7.78  0.0  0.0  1
3  1.51766  13.21  3.69  1.29  72.61  0.57  8.22  0.0  0.0  1
4  1.51742  13.27  3.62  1.24  73.08  0.55  8.07  0.0  0.0  1


In [20]:
X = df.drop(columns=["y"]).values
y = df["y"].values

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (214, 9)
y shape: (214,)


In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (171, 9)
Test shape: (43, 9)


In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("First 5 scaled samples:\n", X_train[:5])

First 5 scaled samples:
 [[-0.84623746 -0.74467528  0.52636164  1.01287057  0.345884    0.33930631
  -0.68807645 -0.33761134 -0.60110996]
 [ 0.28077633  0.33413832  0.54088841 -1.40290866  0.07169648 -0.609516
  -0.03692037 -0.04564469  1.78472688]
 [-0.0992632   0.43938842 -2.03034913 -1.8020374   2.42187517 -0.89595292
   1.44941417 -0.33761134 -0.60110996]
 [ 0.76237816  0.518326   -0.37429784  0.0255521  -0.7900357  -0.01873984
   0.64254685 -0.33761134  1.0888578 ]
 [ 0.03833732 -0.03423706  0.4609912   0.0255521  -0.4375089   0.01706477
  -0.24217609 -0.33761134 -0.60110996]]


In [23]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def predict_proba(X, w, b):
    z = X @ w + b
    return sigmoid(z)

def loss(y, p):
    p = np.clip(p, 1e-10, 1-1e-10)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def update_weights(X, y, w, b, lr):
    p = predict_proba(X, w, b)
    error = p - y
    w = w - lr * (X.T @ error) / len(y)
    b = b - lr * np.mean(error)
    return w, b

In [24]:
w = np.zeros(X_train.shape[1])
b = 0.0
lr = 0.1
epochs = 100

for i in range(epochs):
    p_train = predict_proba(X_train, w, b)
    current_loss = loss(y_train, p_train)
    w, b = update_weights(X_train, y_train, w, b, lr)
    if i % 10 == 0:
        print("Epoch:", i, "Loss:", current_loss)

Epoch: 0 Loss: 0.6931471805599454
Epoch: 10 Loss: 0.6156932528927321
Epoch: 20 Loss: 0.5775659463142817
Epoch: 30 Loss: 0.554707026180174
Epoch: 40 Loss: 0.5392078631294305
Epoch: 50 Loss: 0.527877041659996
Epoch: 60 Loss: 0.5191692332210998
Epoch: 70 Loss: 0.5122334959118788
Epoch: 80 Loss: 0.506557413330293
Epoch: 90 Loss: 0.501811993354646


In [25]:
p_test = predict_proba(X_test, w, b)

print("First 10 probabilities:\n", p_test[:10])

First 10 probabilities:
 [0.46596617 0.02699184 0.60769832 0.02225283 0.30880036 0.30065433
 0.5507444  0.44201251 0.43843704 0.40641316]


In [26]:
def predict_label(p, threshold=0.5):
    return (p >= threshold).astype(int)

y_pred_05 = predict_label(p_test, 0.5)
y_pred_07 = predict_label(p_test, 0.7)

print("Predictions at 0.5 threshold:\n", y_pred_05[:10])
print("Predictions at 0.7 threshold:\n", y_pred_07[:10])

Predictions at 0.5 threshold:
 [0 0 1 0 0 0 1 0 0 0]
Predictions at 0.7 threshold:
 [0 0 0 0 0 0 0 0 0 0]


In [27]:
accuracy = np.mean(y_pred_05 == y_test)
print("Test Accuracy (0.5 threshold):", accuracy)

Test Accuracy (0.5 threshold): 0.8604651162790697
